# UDFs vs Vectorized Pandas UDFs: Architecture & Performance Benchmarks

**Module**: 03_dataframe_and_spark_sql  
**Notebook**: 08_udfs_and_pandas_udfs.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How standard Python UDFs work under Py4J, how Apache Arrow enables Vectorized Pandas UDFs, and measuring performance differences.
- **Why does this topic exist?**: Standard Python UDFs are one of the biggest performance killers in Spark. Vectorized UDFs can be substantially faster for batch-oriented Python logic; the speedup is workload-dependent.
- **What problem does it solve?**: Row-by-row socket serialization between JVM and Python worker processes creates massive CPU and GC overhead.
- **Where is it used in real-world Data Engineering?**: Running machine learning model inference and complex domain logic in PySpark.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | DataFrames, Python, Arrow basics |
| **Required depth** | Working knowledge: can choose built-ins, Python UDFs, or pandas UDFs |
| **Data Engineer** | Use built-ins first, write safe Python UDFs when necessary, and use pandas UDFs for vectorized logic. |
| **Senior Data Engineer** | Diagnose serialization overhead and Arrow-related failures. |
| **Spark specialist** | Explain the Python worker, Py4J, Arrow batches, and codegen boundaries. |
| **Next topic** | `04_execution_engine/01_lazy_evaluation.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A Python UDF is easy to write and expensive to run: every row is serialized between the JVM and a Python
worker, and the optimizer cannot see inside the function. Pandas UDFs batch rows with Arrow and are often much
faster, but they still cross the JVM/Python boundary and add memory pressure. Built-in functions remain the best
default.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Python UDF** | Row-at-a-time Python function | Pickle serialization, no codegen |
| **Pandas UDF** | Vectorized Python function | Arrow batches, pandas Series/DataFrame |
| **Arrow** | Columnar in-memory format | Reduces serialization overhead |
| **Python worker** | Process that runs Python code | Separate from the JVM executor |
| **Determinism** | Same input → same output | Important for retries/lineage |
| **Vectorization** | Process batches, not rows | Better CPU and serialization efficiency |

### How it works, step by step

1. The JVM executor serializes rows or Arrow batches to a Python worker.
2. The Python worker executes the function.
3. Results are serialized back to the JVM.
4. Pandas UDFs use Arrow to transfer columnar batches.
5. The optimizer treats the UDF as a black box.
6. Failures/retries re-run the Python function; non-determinism can change results.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Python UDF vs Pandas UDF Execution Models

```
Standard Python UDF (Row-by-Row Serialization):
[JVM Executor Heap] -> [Socket / Pickle] -> [Python Daemon] -> [Execute Function]
         ^                                                               |
         +------------------- [Socket / Unpickle] <----------------------+
(Severe overhead: 1 serialization event PER ROW; No Catalyst optimization)

Vectorized Pandas UDF (Apache Arrow Shared Memory):
[JVM Executor Heap] ======= Apache Arrow RecordBatch =======> [Python Worker]
                              (Zero-Copy / Bulk SIMD)         [Pandas Series vectorized]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("UDF_Benchmarking_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Implementing Standard Python UDF vs Pandas UDF

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

# 1. Standard Python UDF
@udf(returnType=DoubleType())
def standard_tax_udf(price, quantity):
    if price is None or quantity is None:
        return 0.0
    return float(price * quantity * 0.14)

# 2. Vectorized Pandas UDF (Series to Series)
@pandas_udf(DoubleType())
def vectorized_tax_udf(price: pd.Series, quantity: pd.Series) -> pd.Series:
    return price.fillna(0.0) * quantity.fillna(0) * 0.14

# 3. Native Built-in Function (Baseline)
native_expr = F.round(F.coalesce(F.col("unit_price"), F.lit(0.0)) * F.coalesce(F.col("quantity"), F.lit(0)) * 0.14, 2)

## 4. Benchmark: Native vs Pandas UDF vs Standard Python UDF

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# Duplicate dataset to 240,000 rows to create meaningful timing differences
bench_df = orders.union(orders).union(orders).union(orders).cache()
bench_df.count() # Force cache population

# Benchmark 1: Native Spark Function
t0 = time.time()
bench_df.withColumn("tax", native_expr).write.mode("overwrite").format("noop").save()
t_native = time.time() - t0

# Benchmark 2: Vectorized Pandas UDF (Arrow)
t0 = time.time()
bench_df.withColumn("tax", vectorized_tax_udf(F.col("unit_price"), F.col("quantity"))).write.mode("overwrite").format("noop").save()
t_pandas = time.time() - t0

# Benchmark 3: Standard Python UDF
t0 = time.time()
bench_df.withColumn("tax", standard_tax_udf(F.col("unit_price"), F.col("quantity"))).write.mode("overwrite").format("noop").save()
t_standard = time.time() - t0

print(f"=== Execution Time Benchmark (240k rows) ===")
print(f"1. Native Spark Function: {t_native:.3f}s (Fastest - runs pure JVM)")
print(f"2. Vectorized Pandas UDF: {t_pandas:.3f}s (Arrow Batch Vectorization)")
print(f"3. Standard Python UDF:   {t_standard:.3f}s (Slowest - row-by-row socket pickle)")
print(f"Speedup of Vectorized over Standard: {t_standard / t_pandas:.2f}x")
bench_df.unpersist()

---

## Experiment: Inspecting Whole-Stage Codegen Suppression

- **1. What are we testing?**: Show that standard Python UDF suppresses Whole-Stage Code Generation while Native operations do not.
- **2. What do we expect?**: Native expression shows WholeStageCodegen (*) in explain(); standard UDF drops out of codegen.

In [ ]:
print("=== Native Expression Plan (WholeStageCodegen active) ===")
orders.select(native_expr).explain()

print("\n=== Standard Python UDF Plan (BatchEvalPython operator) ===")
orders.select(standard_tax_udf(F.col("unit_price"), F.col("quantity"))).explain()

**3. What happened?**: The standard UDF introduced BatchEvalPython and disabled WholeStageCodegen.  
**4. Why did it happen?**: Spark must serialize rows out of the JVM into external Python worker processes.  
**5. Practical Engineering Lesson**: Always exhaust native pyspark.sql.functions before resorting to custom Python functions.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: PII masking in a customer dataset

- **Industry**: Healthcare / finance
- **Scenario**: A team masks email addresses and phone numbers before sharing data with analysts.
- **Problem**: Built-in masking functions are limited, and the rule must be applied consistently.
- **Why this concept is relevant**: A deterministic UDF can implement a custom masking policy, but a built-in expression is preferable if it can express the rule.
- **Implementation idea**: Prefer `regexp_replace`/`sha2` built-ins; if a UDF is required, make it deterministic and test it on edge cases.
- **Expected result**: A masked dataset with a documented, repeatable policy.
- **Alternative**: A data governance/masking service at the storage layer.
- **Trade-offs**: Spark masking is portable; a governance service centralizes policy and access control.

### Use Case 2: Feature engineering for a recommendation model

- **Industry**: Recommendation systems
- **Scenario**: A data scientist needs a complex text-similarity feature that is not a Spark built-in.
- **Problem**: A row-at-a-time Python UDF is too slow for hundreds of millions of rows.
- **Why this concept is relevant**: A pandas UDF processes Arrow batches and can use vectorized NumPy/pandas code.
- **Implementation idea**: Group rows into batches, apply the pandas UDF, validate null/edge cases, and compare throughput with a built-in baseline.
- **Expected result**: A faster feature pipeline, with the Python boundary still visible in the plan.
- **Alternative**: A feature store or a model-serving pipeline outside Spark.
- **Trade-offs**: Spark is good for batch feature generation; a feature store is better for online serving.

---

## Business Value

- Can unlock custom logic without abandoning Spark.
- Pandas UDFs can significantly reduce Python serialization overhead.
- Built-in functions reduce cost and improve reliability.
- Clear UDF policy prevents accidental performance regressions.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Prefer built-in functions; measure before and after a UDF.
- Make UDFs deterministic and null-safe.
- Monitor Python worker memory and Arrow batch size.
- Pin pandas/pyarrow versions across driver and executors.
- Test UDF behavior on nulls, empty batches, and malformed input.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Built-in vs Python UDF vs pandas UDF

**Steps**

1. Compute a string transformation with a built-in function, a Python UDF, and a pandas UDF.
2. Time each on the same input.
3. Inspect the physical plan for the UDF nodes.

**What to observe**: Built-ins usually win; pandas UDFs often beat row-at-a-time UDFs but still cross the Python boundary.

**Healthy result**: You can explain the trade-off without claiming a fixed multiplier.

**Problematic result**: A UDF on a hot path dominates runtime and blocks codegen.

**Engineering takeaway**: Use the least expensive abstraction that expresses the logic.

---

## Trade-offs and Alternatives

### Advantages

- Custom logic
- Pandas UDFs are vectorized
- Works with Python ecosystem

### Limitations

- Serialization overhead
- No optimizer visibility
- Version/environment sensitivity

### When to use it

- Custom parsing
- PII masking
- ML features
- Complex string logic

### When not to use it

- Simple logic that built-ins can express
- Non-deterministic UDFs
- Huge Arrow batches without memory planning

### Alternatives inside Spark

- Built-in functions
- SQL expressions
- Scala/Java UDFs

### Alternatives outside Spark

- pandas
- Python services
- Feature stores

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): String Cleansing Pandas UDF
**Problem Statement**: Write a Pandas UDF that takes a Series of strings and strips whitespace and lowercases all characters.  
**Input / Context**: Series of channel strings

In [ ]:
# Exercise 1: Clean string Pandas UDF

### Exercise 2 (Intermediate): Grouped Map Pandas UDF with applyInPandas
**Problem Statement**: Use applyInPandas() to subtract the channel mean unit_price from each individual order within each channel.  
**Input / Context**: orders DataFrame

In [ ]:
# Exercise 2: applyInPandas

### Exercise 3 (Advanced): Scalar Pandas UDF with Multiple Inputs
**Problem Statement**: Implement a Haversine distance or compound interest formula using vectorized NumPy inside a Pandas UDF.  
**Input / Context**: Two numeric columns

In [ ]:
# Exercise 3: Vectorized math

---

## Production Debugging Exercise

### Arrow memory error

**Scenario**: A pandas UDF fails with an Arrow memory or batch-size error.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check `spark.sql.execution.arrow.maxRecordsPerBatch`.
2. Reduce batch size or executor memory pressure.
3. Verify pandas/pyarrow versions on driver and executors.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (String Cleansing Pandas UDF):
```python
@pandas_udf(StringType())
def clean_str_udf(s: pd.Series) -> pd.Series:
    return s.str.strip().str.lower()

orders.select(clean_str_udf(F.col("channel")).alias("clean_channel")).show(3)
```

#### Solution 2 (Grouped Map Pandas UDF with applyInPandas):
```python
def subtract_mean(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf['price_diff'] = pdf['unit_price'] - pdf['unit_price'].mean()
    return pdf[['order_id','channel','unit_price','price_diff']]

res = orders.groupBy("channel").applyInPandas(subtract_mean, schema="order_id int, channel string, unit_price double, price_diff double")
res.show(5)
```

#### Solution 3 (Scalar Pandas UDF with Multiple Inputs):
```python
import numpy as np

@pandas_udf(DoubleType())
def compound_growth(principal: pd.Series, rate: pd.Series) -> pd.Series:
    return principal * np.power(1.0 + rate, 3)

spark.range(3).withColumn("growth", compound_growth(F.lit(100.0), F.lit(0.05))).show()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy of Performance**: 1. Native Functions (Best) > 2. Pandas UDF (Good) > 3. Standard UDF (Avoid).
- **Arrow Zero-Copy**: Pandas UDFs transfer entire batches using Apache Arrow memory format.
- **Whole-Stage Codegen**: Python UDFs break operator fusion across the stage.

### Essential Technical Terms
- `Pandas UDF`: Vectorized UDF using Apache Arrow and Pandas Series.
- `BatchEvalPython`: Physical operator indicating Python socket execution.
- `Apache Arrow`: In-memory columnar data format enabling zero-copy cross-language sharing.

### Common Engineering Mistakes
- **Writing UDFs for existing functions**: e.g. writing a UDF for lower() or round() kills performance.
- **Returning scalar instead of Series**: A Series-to-Series Pandas UDF must accept and return pd.Series of identical length.

### Interview Questions on This Topic
- **Q: Why are standard Python UDFs slow in PySpark?**
  - *A*: They require row-by-row serialization across a socket between the JVM and Python worker, disabling Catalyst optimizations and Whole-Stage Codegen.
- **Q: Why can Pandas UDFs be much faster than row-at-a-time UDFs?**
  - *A*: They use Apache Arrow to transfer entire columnar record batches directly into memory without row-level serialization overhead.


### More Interview Questions

- **Q: What is the main cost of a Python UDF?**
  - *A*: Row-by-row serialization between the JVM and Python worker, plus the optimizer's inability to inspect the function.
- **Q: How does a pandas UDF reduce that cost?**
  - *A*: It transfers Arrow columnar batches and processes many rows at once, reducing per-row serialization and Python overhead.

### What to Study Next
Module 03 complete! Proceed to Module 04: Spark Execution Engine Deep Dive.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')